# Constructing simulation scenes
Simulation is an important component in the robotics toolkit. It allows us to test our algorithms in a safe and repeatable environment. In this homework, we will learn how to construct a simulation scene using semantics from our `airo-mono` repo which uses the Drake rigid body simulator. We will construct a scene with a table, rack, a robot arm, and a gripper. We will also learn how to construct frames for planning the end-effector of the robot arm. 
    

In [ ]:
!current_env=$(conda info --envs | grep "*" | awk '{print $1}'); if [ "$current_env" = "irm" ]; then echo "The current active Conda environment is 'irm'. You can run the next cell."; else echo "The current active Conda environment is not 'irm'. Make sure to run this notebook with your irm conda env (conda activate irm)"; fi

In [ ]:
import math
import os
import sys
from pathlib import Path

import airo_models
from airo_drake import finish_build
from airo_drake import visualize_frame
from airo_models.urdf import replace_value, make_static
from loguru import logger
from pydrake.geometry import MeshcatVisualizer, MeshcatVisualizerParams, Role
from pydrake.math import RollPitchYaw
from pydrake.multibody.tree import FixedOffsetFrame
from pydrake.planning import RobotDiagramBuilder

# Change the minimum logging level
logger.remove()
logger.add(sys.stdout, format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}",
           filter=lambda record: record["level"].name == "INFO")

import numpy as np
from pydrake.geometry import Meshcat
from pydrake.math import RigidTransform, RotationMatrix

# Some helper functions. You can ignore these. 

def add_meshcat(robot_diagram_builder: RobotDiagramBuilder, meshcat=None) -> Meshcat:
    scene_graph = robot_diagram_builder.scene_graph()
    builder = robot_diagram_builder.builder()

    # Adding Meshcat must be done before finalizing
    meshcat = Meshcat() if meshcat is None else meshcat
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

    # Add visualizer for proximity/collision geometry
    collision_params = MeshcatVisualizerParams(role=Role.kProximity, prefix="collision", visible_by_default=False)
    MeshcatVisualizer.AddToBuilder(builder, scene_graph.get_query_output_port(), meshcat, collision_params)

    return meshcat


def _make_static(urdf):
    joint_types = ["revolute", "continuous", "prismatic", "floating", "planar"]
    for joint_type in joint_types:
        replace_value(urdf, "@type", joint_type, "fixed")
    return urdf

In [ ]:
# setup visualization
meshcat = Meshcat()

## Building simulation environments

There are many (robot) simulators, generally known as multi-body simulators, available. Building an environment (which we sometimes call a scene) generally follows the next steps:
1. Initialize an empty environment, parsers and other utilities (e.g. visualizations)
2. Collect your models: this can be filepaths to 3D models or text files containing the objects morphology (for example URDF files, which we will introduce later in this notebook)
3. Add these models to your simulator
4. Get mounting and attachment frames of your models
5. Specify child-parent transforms (e.g. the robot is mounted 2.4 cm above the table)
6. Weld child and parent frames together

We will walk you through the different objects in this scene. The complete tree of the scene we will build in this homework looks like this:

world&emsp;->&emsp;table&emsp;->&emsp;mounting_plate&emsp;->&emsp;arm&emsp;->&emsp;gripper

&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;->&emsp;rack 

First, we consider the table on which your robot and the other objects are mounted. We give a simple example of putting a table in the world in the next code cell. You do not have to know these semantics, they follow an API of a specific simulator (drake in this case). However, it is useful to glance over the code and use it as an example in future exercises since we will be using simulation for the rest of the IRM course. 



In [ ]:
# Init env
robot_diagram_builder = RobotDiagramBuilder()
plant = robot_diagram_builder.plant()
parser = robot_diagram_builder.parser()
parser.SetAutoRenaming(True)

meshcat.Delete()
meshcat.DeleteAddedControls()
add_meshcat(robot_diagram_builder, meshcat)

# 1. Find your models
table_urdf_path = str(Path(os.path.join("resources", "table.urdf")).resolve())

# 2. Add models to simulator
table_index = parser.AddModels(table_urdf_path)[0]

# 3. Get mounting and attach frames for all models
world_frame = plant.world_frame()
table_frame = plant.GetFrameByName("base_link", table_index)

# 4. Define relative transform, i.e. specify child-parent transforms
xyz_world_table = [0, 0, 0]
X_World_Table = RigidTransform(RollPitchYaw(0, 0, 0), xyz_world_table)

# 5. Weld frames together
plant.WeldFrames(world_frame, table_frame, X_World_Table)

# Finish the build of the simulation env
robot_diagram, context = finish_build(robot_diagram_builder, meshcat)
robot_diagram.ForcedPublish(context)

<div class="alert alert-block alert-info"> 
Go to your meshcat instance in browser (http://localhost:7000) and find the table.
</div>

<a id="specs-semantics"></a>
Let's go over some of the crucial variables and concepts involved in mounting our table:
- **urdf**: the filepath of the URDF file. URDF is an alternative notation to DH notation to describe physical objects. We will explain this in the next section
- **Child parent relation**: describes how the table will be attached to the parent. Here, we attach the root link of the table to the world. This implies that the origin of our table lies at the origin of the world. Hence, *the origin of the table is the origin of the world*. This is an important notion. The choice of world coordinate system is task-specific. In this case we find the choosing the table as origin of our simulator useful so we can measure everything relative to the table.
- **WeldFrame(parent, child, X_P_C)**: attaches the child to the parent with a relative transform between parent and child specified as `X_Parent_Child` (shorthand as `X_P_C`)

The table is added in the simulator with the `parser.AddModels()` call, but is only spawned later, simultaneously with all other required objects when calling `finish_build()`. This is because the simulator we use (Drake) requires us to spawn all objects at once, for optimization reasons. 

### Positioning the mounting plate
We will determine the position of the mounting plate of the robot on the table. We can specify the table position relative to any object. For convenience, we will express its position relative to the origin of the table. The origin of the table is in the left upper corner when we are standing in front of the table. 

<div class="alert alert-block alert-info"> 
    <b>Task 1</b>:
    Fill in the position of the mounting plate relative to the table origin. We will assume that the mounting plate is horizontally 18 cm offsetted and vertically 22 cm. There is no height offset required. The following picture shows the coordinate system of the origin of the table (i.e., the frame in the left top corner). 
</div>

![table origin](resources/triad-table.jpeg)

In [ ]:
#TODO (@STUDENTS): complete the following
x_offset = 0.0
y_offset = 0.0

In [ ]:
rpy_table_mountingplate = [0, 0, 0]
xyz_table_mountingplate = [x_offset, y_offset, 0.0]

X_table_mountingplate = [rpy_table_mountingplate, xyz_table_mountingplate]

In [ ]:
# Init env
robot_diagram_builder = RobotDiagramBuilder()
plant = robot_diagram_builder.plant()
parser = robot_diagram_builder.parser()
parser.SetAutoRenaming(True)

meshcat.Delete()
meshcat.DeleteAddedControls()
add_meshcat(robot_diagram_builder, meshcat)

# 1. Find your models
table_urdf_path = str(Path(os.path.join("resources", "table.urdf")).resolve())
mounting_plate_urdf_path = str(Path(os.path.join("resources", "mounting_plate.urdf")).resolve())

# 2. Add models to simulator
table_index = parser.AddModels(table_urdf_path)[0]
mounting_plate_index = parser.AddModels(mounting_plate_urdf_path)[0]

# 3. Get mounting and attach frames for all models
world_frame = plant.world_frame()
table_frame = plant.GetFrameByName("base_link", table_index)
mounting_plate_frame = plant.GetFrameByName("base_link", mounting_plate_index)

# 4. Define relative transform, i.e. specify child-parent transforms
X_Table_MountingPlate = RigidTransform(RollPitchYaw(*rpy_table_mountingplate), xyz_table_mountingplate)

# 5. Weld frames together
plant.WeldFrames(world_frame, table_frame)
plant.WeldFrames(table_frame, mounting_plate_frame, X_Table_MountingPlate)

# Finish the build of the simulation env
robot_diagram, context = finish_build(robot_diagram_builder, meshcat)
robot_diagram.ForcedPublish(context)

## URDF: an alternative notation to DH notation
The Universal Robot Description Format (URDF) is an XML format for representing a robot model. It is a widely used format to describe robots and objects in their environments. It originated in the ROS community, a set of open-source software libraries and tools for building robot applications. Compared to the DH notation, URDF is richer in it specification: you can describe kinematics, intertial properties and link geometry of a robot. The full specification can be found [here](http://wiki.ros.org/urdf/XML).

The two main elements building up an URDF file are links and joints. Links are rigid bodies and joints are the connections between links. A link can have a visual and a collision element. This distinction is made to allow for more efficient collision checking. Often, we choose a simplified geometry for the collision element (the UR3e robot for example is simplified by stitching spheres and cylinders).
The joints connect two links: a parent and a child link. The joints can be of different types: fixed, revolute, prismatic, continuous, floating, planar, and fixed. The most common ones we will use are fixed (0 DOF) and revolute (1 DOF) joints. You can model any robot/physical object with a tree structure like serial-chain robot arms. This implies that modelling closed-loop kinematic chains is not possible!

There are some conventions we follow in our URDF files and 3D models. These are originated to avoid introspecting the URDF XML files and allow us to deploy them more broadly without friction. Some conventions:  
- the starting link in a URDF model is always called `base_link`. This can be a virtual link (i.e. no geometry). 
- we assume everything is modelled and defined with the X+ axis forward.  
- provide handy virtual frames. For example, our [table urdf](resources/table.urdf) has a virtual link called `table_centre` which is the centre-top of the table. You can use this to fix objects at the top of the table. Not providing this frame would require you to compute the transform from the origin of the table to the top of the table (i.e. translate 2 cm in Z direction).

An example where we apply these conventions is the [mounting plate](resources/mounting_plate.urdf) of the robot. Notice the starting link is called `base_link` and the virtual link `plate_top_centre` is defined. We model the mounting plate 6mm underneath its base link. This way, we can attach the mounting plate to the table without offsetting in the Z direction (see the joint `base_link_2_mounting_plate`).

Note that these conventions are not enforced by the URDF specification, nor are they universally followed. 

As you have noted from the mentioned examples, it is not required to always model robots with URDF; we can also model plain objects with URDF. For example, we defined a table in URDF [here](resources/table.urdf). You will model a rack in URDF in the next section. We will later assemble all predefined objects into the simulator by welding the associated parent and child frames of the objects together. 
 
URDF files are XML files without any macros, constants, parametrization or math capabilities. To enhance the format, the ROS community introduced [Xacro](http://wiki.ros.org/urdf/Tutorials/Using%20Xacro%20to%20Clean%20Up%20a%20URDF%20File): a macro language that allows you to write more compact URDF files by adding basic programming capabilities. For example, one can parametrize the dimensions of the [mounting plate](resources/mounting_plate.urdf). We will not use Xacro in this course as it requires a functional ROS installation. 

An example of a simple cube, attached to a cylinder in URDF notation:
```xml
<?xml version="1.0"?>
<robot name="cube_and_cylinder">
  <!-- Define the links -->
  <link name="base_link"/>

  <link name="cylinder_link">
    <visual>
      <geometry>
        <cylinder radius="0.05" length="0.2"/>
      </geometry>
    </visual>
    <collision>
      <geometry>
        <cylinder radius="0.05" length="0.2"/>
      </geometry>
    </collision>
    <inertial>
      <mass value="1.0"/>
      <inertia ixx="0.01" ixy="0.0" ixz="0.0" iyy="0.01" iyz="0.0" izz="0.01"/>
    </inertial>
  </link>

  <link name="cube_link">
    <visual>
      <geometry>
        <box size="0.1 0.1 0.1"/>
      </geometry>
    </visual>
    <collision>
      <geometry>
        <box size="0.1 0.1 0.1"/>
      </geometry>
    </collision>
    <inertial>
      <mass value="1.0"/>
      <inertia ixx="0.01" ixy="0.0" ixz="0.0" iyy="0.01" iyz="0.0" izz="0.01"/>
    </inertial>
  </link>

  <!-- Define the joints -->
  <joint name="base_to_cylinder" type="fixed">
    <parent link="base_link"/>
    <child link="cylinder_link"/>
    <origin rpy="0 0 0" xyz="0 0 0.1"/>
  </joint>

  <joint name="cylinder_to_cube" type="fixed">
    <parent link="cylinder_link"/>
    <child link="cube_link"/>
    <origin rpy="0 0 0" xyz="0 0 0.2"/>
  </joint>

</robot>
```



The robot arms (UR3e) and grippers (Robotiq 2f 85) we use in this course are also modelled in URDF notation. You do not find them in the resources folder because they are contained in our `airo-models` python module. 

## Modeling a rack in URDF

<div class="alert alert-block alert-info" > 
    <b>Task 2</b>:
    Complete in the URDF file rack.urdf to make a rack. It should consist of a backplate and two shelves. We already provided the backplate and lower shelf. Your task is to make the upper shelf. It should have the same dimensions as the lower shelf. Mount the upper shelf at the top of backplate. You can verify your solution by running the next cell. Press (ESC) if you still see the table in Drake MeshCat.    
</div>

In [ ]:
from pydrake.visualization import ModelVisualizer

print(
    f"View your URDF model on {meshcat.web_url()}. Press ESCAPE to stop visualizing and return control to this notebook.")

viewer = ModelVisualizer(meshcat=meshcat)
viewer.AddModels(os.path.join("resources", "rack.urdf"))
viewer.Run()

## Mounting the rack on the table

<div class="alert alert-block alert-info" > 
    <b>Task 3</b>:
    Mount your rack on the table in a pose reachable for the robot. It is possible that you want to both change the position and orientation of the rack on the table. You can change the orientation by expressing it as orientations around the principal axes (roll around X-axis, pitch around Y-axis and yaw around Z-axis). Note that we are positioning the rack relative to the origin coordinate system of the table (see the picture of task 1).
</div>

In [ ]:
#TODO (@STUDENTS): complete the following code
x_offset = 0.
y_offset = 0.
z_offset = 0. # rack_height = 0.536

roll = 0
pitch = 0
yaw = 0

X_Table_MountingRack = RigidTransform(RollPitchYaw(roll, pitch, yaw), [x_offset, y_offset, z_offset])

In [ ]:
# Init env
robot_diagram_builder = RobotDiagramBuilder()
plant = robot_diagram_builder.plant()
parser = robot_diagram_builder.parser()
parser.SetAutoRenaming(True)

meshcat.Delete()
meshcat.DeleteAddedControls()
add_meshcat(robot_diagram_builder, meshcat)

# 1. Find your models
table_urdf_path = str(Path(os.path.join("resources", "table.urdf")).resolve())
mounting_plate_urdf_path = str(Path(os.path.join("resources", "mounting_plate.urdf")).resolve())
arm_urdf_path = airo_models.get_urdf_path("ur3e")
gripper_urdf_path = airo_models.get_urdf_path("robotiq_2f_85")
gripper_urdf = airo_models.urdf.read_urdf(gripper_urdf_path)
make_static(gripper_urdf)
gripper_urdf_path = airo_models.urdf.write_urdf_to_tempfile(
    gripper_urdf, gripper_urdf_path, prefix="robotiq_2f_85_static_"
)
rack_urdf_path = str(Path(os.path.join("resources", "rack.urdf")).resolve())

# 2. Add models to simulator
table_index = parser.AddModels(table_urdf_path)[0]
mounting_plate_index = parser.AddModels(mounting_plate_urdf_path)[0]
arm_index = parser.AddModels(arm_urdf_path)[0]
gripper_index = parser.AddModels(gripper_urdf_path)[0]
rack_index = parser.AddModels(rack_urdf_path)[0]

# 3. Get mounting and attach frames for all models
world_frame = plant.world_frame()
table_frame = plant.GetFrameByName("base_link", table_index)
mounting_plate_frame = plant.GetFrameByName("base_link", mounting_plate_index)
mounting_plate_top_centre_frame = plant.GetFrameByName("plate_top_centre",
                                                       mounting_plate_index)  # this is a virtual helper frame 
arm_frame = plant.GetFrameByName("base_link", arm_index)
arm_tool_frame = plant.GetFrameByName("tool0", arm_index)
gripper_frame = plant.GetFrameByName("base_link", gripper_index)
rack_frame = plant.GetFrameByName("base_link", rack_index)

# 4. Define relative transform, i.e. specify child-parent transforms
X_Table_MountingPlate = RigidTransform(RollPitchYaw(*rpy_table_mountingplate), xyz_table_mountingplate)
X_MountingPlateTopCentre_ArmBaseLink = RigidTransform(RollPitchYaw(0, 0, -np.pi / 2), [0, 0, 0])
X_Tool0_GripperBase = RigidTransform(RollPitchYaw(0, 0, 0), [0, 0, 0])
# X_Table_MountingRack is specified above
# Add TCP frame
tcp_offset = 0.174
# TCP is between the fingers and tcp_offset cm from the wrist towards the gripper forwards direction (Z+ in TCP frame)
X_Tool0Tcp = RigidTransform(RotationMatrix.Identity(), [0, 0, tcp_offset])
arm_tcp_frame = plant.AddFrame(FixedOffsetFrame("tcp", plant.GetFrameByName("tool0", arm_index), X_Tool0Tcp))

# 5. Weld frames together
plant.WeldFrames(world_frame, table_frame)  # world -> table
plant.WeldFrames(table_frame, mounting_plate_frame, X_Table_MountingPlate)  # table -> mountingplate
plant.WeldFrames(mounting_plate_top_centre_frame, arm_frame,
                 X_MountingPlateTopCentre_ArmBaseLink)  # mountingplate -> arm
plant.WeldFrames(arm_tool_frame, gripper_frame, X_Tool0_GripperBase)  # arm -> gripper 
plant.WeldFrames(table_frame, rack_frame, X_Table_MountingRack)  # table -> rack

# Finish the build of the simulation env
robot_diagram, context = finish_build(robot_diagram_builder, meshcat)
robot_diagram.ForcedPublish(context)

An URDF model can only have a single root link implying that we only model a single object with one URDF file. To assemble multiple models into one simulation environment, we use the API of the simulator Drake. Open-source alternatives exist such as the [SDF format](http://sdformat.org/) that allows describing individual objects and their relationships in the environment. Unfortunately, the SDF format is not feature-complete and has limited downstream support projects which in turn do not implement the whole specification. 

## Expressing a frame relative to the lower shelf

For the following task, we want to express a TCP frame that will grasp above the center of the lower shelf. First, take a look at the coordinate system origin of the lower shelf by running the next cell and examining your meshcat tab. 

In [ ]:
meshcat.Delete("/scenario")
X_W_LowerShelfOrigin = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(),
                                                 plant.GetBodyByName("lower_shelf_link",
                                                                     plant.GetModelInstanceByName("wooden_rack")))
visualize_frame(meshcat, "/scenario/lower_shelf_origin", X_W_LowerShelfOrigin, length=0.15, radius=0.005)

<div class="alert alert-block alert-info" > 
    <b>Task 4</b>:
    Given X_W_LowerShelfOrigin, find X_W_TCP such that the fingers of the gripper will be reaching into the shelf (fingers forward = Z forward), 18 cm above the center of the lower shelf. 
</div>

You can check the result yourselves in the meshcat instance on your browser. It might be that you cannot find the drawn coordinate system because it is occluded by other objects. Disable the `visualizer` and `illustration` in the scene browser to disable the meshes from the objects.

In [ ]:
X_W_TCP = np.zeros((4, 4))
X_W_LowerShelfOrigin = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(),
                                                 plant.GetBodyByName("lower_shelf_link", plant.GetModelInstanceByName(
                                                     "wooden_rack"))).GetAsMatrix4()
# TODO(@Student): Modify (rotate and translate) X_W_TCP using X_W_LowerShelfOrigin to complete Task 4
visualize_frame(meshcat, "/scenario/X_W_TCP", X_W_TCP, length=0.15, radius=0.005, )

## THE END

Now you are ready for practical session 3. In this session you will use your UR3e robot to complete a pick-and-place task in your newly created environment!